# 수정본의 실행·해석 범위

이 노트북은 2026-10-07 코드 수정본입니다. 저장된 출력과 과거 성능 숫자는 제거했습니다. 실제 Excel 전체 처리, 모델 학습·탐색과 성능 재평가는 이번 수정에서 실행하지 않았습니다. 노트북 폴더를 작업 디렉터리로 사용하고 원자료와 `ml_validation.py`, 프로젝트 의존성을 준비한 뒤 위에서 아래로 실행해야 합니다.

`수익률`은 기존 Excel 값을 신뢰하지 않고 매수가·매도가·구매주식수·투자원금으로 다시 계산합니다. 수수료·세금·배당을 포함하지 않은 계산입니다. 이전에 열람한 2021년 이후 자료는 **과거 자료 재평가**이며, 새로 봉인된 최종 테스트가 아닙니다. 전년도 재무제표라는 이유만으로 매수 당시 공개된 정보라고 보장할 수 없으므로 공시 시점 자료의 연결은 별도 과제입니다. 입력 계약과 한계는 `NOTEBOOK-REPAIR-NOTES.md`를 확인하세요.

현재 보관된 `ALL_profit_final(개).xlsx`의 13,979행 중 3행은 매도가가 없습니다(Excel 행 942, 952, 2536). 이 파일을 그대로 읽으면 `recalculate_returns`가 의도적으로 오류를 냅니다. 결측을 0으로 바꾸거나 행을 자동 삭제해 학습을 진행하지 않습니다. 원가격·보유기간 자료를 확인해 처리 근거를 정하기 전에는 전체 학습을 재개할 수 없습니다. 초기 회계 진단 비교에서 유효 13,976행의 포지션 손익 부호 141개가 양수에서 비양수로 바뀌었습니다. 이 수치는 새 가격상승 정답의 변경 개수와 다릅니다. 이는 모델 성능 검증이 아닙니다. 근거: `saved-label-audit.json`.

## 분류 목표를 가격 상승으로 분리했습니다

현재 목표는 **수정주가 기준 가격 상승 여부**입니다. `add_price_direction_targets(recalculate_returns(frame))`로 회계 열을 유지한 뒤 `가격변화율 = 100 × (매도가 / 매수가 − 1)`, `가격상승여부 = int(매도가 > 매수가)`를 추가합니다. 모델의 정답 y는 `가격상승여부`이고, `수익률_양음`은 배분된 원금·정수 주식 수에 따른 회계 손익 부호로 보존합니다.

원자료의 0주 행 25개 중 가격이 상승한 8행은 포지션 이익 0이지만 가격상승 정답은 1입니다(`verification/zero-share-audit.json`). 수정주가는 당시 체결 가능한 실제 호가와 같다고 보장되지 않습니다. 당시 가격 없이 수정주가로 정수 주식 수를 계산한 백테스트의 현실성, 원자료의 주식분할·배당 등 보정 정의는 아직 확인하지 않았습니다. 따라서 새 목표는 기업 자체의 성장·투자 수익의 증명이 아니라 저장된 수정주가의 기간 변화입니다. 기존 포지션 이익 분류의 성능 숫자를 새 목표의 성능으로 재사용하지 않습니다.

2026-10-08: 연도 키로 지표를 맞춰 평균하고 결측 지표는 선별하지 않습니다. 각 연도 회사당 500만 원으로 만든 독립 학습용 관측이며 연속 재투자 성과가 아닙니다. 매도 결과 결측은 여전히 중단합니다. Q1 가격은 두 끝 관측값 평균입니다.


In [ ]:
import KSE_가치팩터_functions as kse_functions
import pandas as pd
import numpy as np
from ml_validation import recalculate_returns, add_price_direction_targets

In [ ]:
### 수정주가 데이터프레임

In [ ]:
# 엑셀 파일 경로 (파일 경로는 실제 경로로 수정해야 합니다)
file_path = 'KSE_FIN_DATA_2023.xlsx'

# 전체 데이터셋 로드 (파일 경로와 시트 이름을 실제 데이터에 맞게 수정하세요)
df_close = pd.read_excel(file_path, sheet_name='수정주가')

# 날짜 열을 datetime 형태로 변환
df_close['Symbol Name'] = pd.to_datetime(df_close['Symbol Name'])

In [ ]:
### 1월 첫 관측과 3월 마지막 관측 수정주가의 평균(일별 평균 아님)

In [ ]:
#기업 리스트 작성
company_columns = [col for col in df_close.columns if col != 'Symbol Name']

# 연도별 평균 주가를 저장할 빈 데이터 프레임 생성
average_prices_df = pd.DataFrame()

# 각 연도에 대해 평균 주가 계산 및 데이터 프레임에 추가
for year in range(2000, 2024):  # 연도 범위는 실제 데이터에 맞게 조정하세요
    q1_averages = kse_functions.calculate_q1_endpoint_mean(df_close, year, company_columns)
    q1_averages['Year'] = year
    average_prices_df = pd.concat([average_prices_df, pd.DataFrame([q1_averages])], ignore_index=True)
    
# 'Year' 열을 데이터 프레임의 첫 번째 열로 이동
average_prices_df = average_prices_df[['Year'] + [col for col in average_prices_df.columns if col != 'Year']]

average_prices_df['Year'] = average_prices_df['Year'].astype(int)

# 결과 데이터 프레임 확인
average_prices_df

In [ ]:
### PER

In [ ]:
### EPS 추출

In [ ]:
# 엑셀 파일 읽기df_fin = pd.read_excel(file_path)
df_fin = pd.read_excel(file_path)

# Name과 EPS 컬럼을 추출하고, 회계년으로 정렬
EPS_Pure_df = df_fin[['Name', '회계년', 'EPS']].sort_values(by=['회계년'])

# 회계년을 Year로 통일하여 전처리
EPS_Pure_df.rename(columns={'회계년': 'Year'}, inplace=True)

EPS_Pure_df

In [ ]:
### PER 계산 (추후 설명변수로 들어갈꺼니까 가공 준비)

In [ ]:
# EPS 데이터를 연도별로 재구조화
EPS_Pure_df_pivot = EPS_Pure_df.pivot(index='Year', columns='Name', values='EPS')

# PER 데이터 프레임 초기화
df_per = pd.DataFrame()

# 연도별로 PER 계산
for year in range(EPS_Pure_df['Year'].min() + 1, average_prices_df['Year'].max() + 1):
    if year in average_prices_df['Year'].values and (year - 1) in EPS_Pure_df_pivot.index:
        previous_year_eps = EPS_Pure_df_pivot.loc[year - 1]  # 이전 연도의 EPS
        current_year_prices = average_prices_df[average_prices_df['Year'] == year].iloc[0]  # 해당 연도의 평균 주가

        # PER 계산
        per_data = {'Year': year}
        for company in company_columns:
            eps = previous_year_eps.get(company)
            price = current_year_prices.get(company)
            if eps and price and eps != 0:
                per = price / eps
            else:
                per = None  # EPS가 0이거나 없는 경우 PER 계산 불가
            per_data[company] = per

        # PER 데이터 추가
        df_per = pd.concat([df_per, pd.DataFrame([per_data])], ignore_index=True)

# Year의 형을 정수형로 바꾸기
df_per['Year'] = df_per['Year'].astype(int)

# 결과 출력
df_per

In [ ]:
### 1/PER 계산

In [ ]:
# df_per 데이터 프레임의 복사본 생성
df_inverse_per = df_per.copy()

# 모든 기업에 대해 1/PER 계산
for company in company_columns:
    df_inverse_per[company] = 1 / df_inverse_per[company]

# 결과 출력
df_inverse_per

In [ ]:
### 기업의 수

In [ ]:
num_company = len(company_columns)

top_per_year = pd.DataFrame()

for year in range(2001, 2024):
    top_companies = kse_functions.top_companies_by_inverse_Rank(df_inverse_per, year, num_company) # 매개변수를 조절함으로 조절 가능
    top_companies['Year'] = year
    top_companies['Rank'] = range(1, len(top_companies) + 1)
    top_per_year = pd.concat([top_per_year, top_companies], ignore_index=True)

# 결과 출력
top_per_year

In [ ]:
close_1 = df_close.copy()

# 첫 번째 열을 인덱스로 설정
close_1.set_index(close_1.columns[0], inplace=True)

# 인덱스를 날짜 형식으로 변환
close_1.index = pd.to_datetime(close_1.index)

# 초기 설정
sale_month_imsi = 3
sale_year_imsi = 2002
purchase_month_imsi = 4 
purchase_year_imsi = 2001

# 시작 연도와 종료 연도 설정
start_year = 2001
end_year = 2022

# 각 연도에 대한 처리
total_values = {}
profits = {}

for year in range(start_year, end_year + 1):

    top_per_current_year = top_per_year[top_per_year['Year'] == year]
    Purchase_current = kse_functions.Price_for_Purchase_Sale(purchase_year_imsi + (year - start_year), purchase_month_imsi, 
                                               sale_year_imsi + (year - start_year), sale_month_imsi,
                                               top_per_current_year, close_1)

    if year == start_year:
        share_current = kse_functions.Purchase_share(Purchase_current, top_per_current_year, 5000000 * len(top_per_current_year))
    else:
        share_current = kse_functions.Purchase_share(Purchase_current, top_per_current_year, 5000000 * len(top_per_current_year))

    Purchase_current_first = Purchase_current.drop_duplicates(subset=['기업명', '매수일'], keep='first')
    Purchase_current_share = pd.merge(share_current, Purchase_current_first, on='기업명', how='left')

    # 잔금 계산 및 '잔금' 컬럼 추가
    # 잔금은 총 매출에 포함되어 있으므로 전체 투자원금을 차감합니다.
    Purchase_current_share = add_price_direction_targets(recalculate_returns(Purchase_current_share))

    # 포트폴리오의 총 가치 계산
    total_value_current = Purchase_current_share['매출'].sum()
    total_values[year] = total_value_current
    profits[year] = Purchase_current_share

# 결과 출력 (필요에 따라 조정)
for year, profit in profits.items():
    print(f"Year: {year}, Total Value: {total_values[year]}")
    print(profit)

In [ ]:
# 각 연도에 대한 처리
for year in range(start_year, end_year + 1):
     # 연도별 Top 20 주식 선정
    top_per_current_year = top_per_year[top_per_year['Year'] == year]
    
    # 구매 및 판매 가격 계산
    Purchase_current = kse_functions.Price_for_Purchase_Sale(
        purchase_year_imsi + (year - start_year), purchase_month_imsi, 
        sale_year_imsi + (year - start_year), sale_month_imsi,
        top_per_current_year, close_1
    )

    # 주식 분배 계산
    if year == start_year:
        share_current = kse_functions.Purchase_share(Purchase_current, top_per_current_year, 5000000 * len(top_per_current_year))
    else:
        share_current = kse_functions.Purchase_share(Purchase_current, top_per_current_year, 5000000 * len(top_per_current_year))

    # 구매 데이터 정리
    Purchase_current_first = Purchase_current.drop_duplicates(subset=['기업명', '매수일'], keep='first')
    Purchase_current_share = pd.merge(share_current, Purchase_current_first, on='기업명', how='left')

    # 잔금 및 수익 계산
    # 잔금은 총 매출에 포함되어 있으므로 전체 투자원금을 차감합니다.
    Purchase_current_share = add_price_direction_targets(recalculate_returns(Purchase_current_share))

    # 포트폴리오의 총 가치 계산 및 저장
    total_value_current = Purchase_current_share['매출'].sum()
    total_values[year] = total_value_current
    profits[year] = Purchase_current_share
    # 연도별 데이터 프레임 저장 확인
    if isinstance(Purchase_current_share, pd.DataFrame):
        profits[year] = Purchase_current_share
    else:
        print(f"Year {year} did not produce a valid DataFrame")

# 각 연도별 데이터 프레임에 '연도' 열 추가 및 리스트에 저장
all_profits = []
for year, profit_df in profits.items():
    if isinstance(profit_df, pd.DataFrame):
        profit_df['연도'] = year
        all_profits.append(profit_df)
    else:
        print(f"Invalid DataFrame for year {year}")

# 모든 연도의 데이터 프레임을 하나로 결합
if all_profits:
    combined_profit = pd.concat(all_profits)
    # 결과 출력
    print(combined_profit)
else:
    print("No valid DataFrames to combine")

In [ ]:
output_filename = "PER_profit_for_tree.xlsx"
combined_profit.to_excel(output_filename, index=False)

print(f"Data saved to {output_filename}")

연도는 앞선 반복문에서 붙입니다. `recalculate_returns`가 회계 손익·수익률·수익률_양음을 계산하고, `add_price_direction_targets`가 별도 가격변화율·가격상승여부를 추가합니다. Excel에서 정답을 수동 작성하지 않으며 이전 중간 파일을 읽을 때도 두 계산을 순서대로 적용합니다.

In [ ]:
df_per_profit = add_price_direction_targets(recalculate_returns(pd.read_excel("PER_profit_for_tree.xlsx")))

# df_per_profit과 df_per 데이터프레임을 병합하는 함수
def merge_per_values(df_profit, df_inverse_per):
    # 새로운 데이터프레임 생성
    df_merged = df_profit.copy()

    # 연도와 기업명을 기준으로 1/PER 값을 매핑
    for index, row in df_profit.iterrows():
        year = row['연도']
        company = row['기업명']
        
        # df_inverse_per에서 해당 연도와 기업의 PER 값을 찾음
        per_value = df_inverse_per.loc[df_inverse_per['Year'] == year, company].iloc[0]

        # 새로운 열에 값을 추가
        df_merged.at[index, '1/PER'] = per_value

    return df_merged

# 함수를 사용하여 df_per_profit과 df_per 병합
df_merged = merge_per_values(df_per_profit, df_inverse_per)

# 결과 확인
df_merged

In [ ]:
# 원본 데이터프레임의 복사본을 생성
df_per_profit_cleaned = df_merged.copy()

# NaN 값이 있는 행을 제거
df_per_profit_cleaned.dropna(subset=['1/PER'], inplace=True)

# 1/PER 값의 순위를 내림차순으로 계산
df_per_profit_cleaned['1/PER Rank'] = df_per_profit_cleaned.groupby('연도')['1/PER'].rank(ascending=False)

# 결과 확인
df_per_profit_cleaned

In [ ]:
output_filename = "PER_profit_final.xlsx"
df_per_profit_cleaned.to_excel(output_filename, index=False)

print(f"Data saved to {output_filename}")

In [ ]:
### PBR

In [ ]:
# 엑셀 파일 읽기df_fin = pd.read_excel(file_path)
df_fin = pd.read_excel(file_path)

# Name과 BPS 컬럼을 추출하고, 회계년으로 정렬
BPS_Pure_df = df_fin[['Name', '회계년', 'BPS']].sort_values(by=['회계년'])

# 회계년을 Year로 통일하여 전처리
BPS_Pure_df.rename(columns={'회계년': 'Year'}, inplace=True)

BPS_Pure_df

In [ ]:
# BPS 데이터를 연도별로 재구조화
BPS_Pure_df_pivot = BPS_Pure_df.pivot(index='Year', columns='Name', values='BPS')

# PBR 데이터 프레임 초기화
df_pbr = pd.DataFrame()

# 연도별로 PBR 계산
for year in range(BPS_Pure_df['Year'].min() + 1, average_prices_df['Year'].max() + 1):
    if year in average_prices_df['Year'].values and (year - 1) in BPS_Pure_df_pivot.index:
        previous_year_bps = BPS_Pure_df_pivot.loc[year - 1]  # 이전 연도의 BPS
        current_year_prices = average_prices_df[average_prices_df['Year'] == year].iloc[0]  # 해당 연도의 평균 주가

        # PBR 계산
        pbr_data = {'Year': year}
        for company in company_columns:
            bps = previous_year_bps.get(company)
            price = current_year_prices.get(company)
            if bps and price and bps != 0:
                pbr = price / bps
            else:
                pbr = None  # BPS가 0이거나 없는 경우 PBR 계산 불가
            pbr_data[company] = pbr

        # PBR 데이터 추가
        df_pbr = pd.concat([df_pbr, pd.DataFrame([pbr_data])], ignore_index=True)

# Year의 형을 정수형로 바꾸기
df_pbr['Year'] = df_pbr['Year'].astype(int)

# 결과 출력
df_pbr

In [ ]:
# df_pbr 데이터 프레임의 복사본 생성
df_inverse_pbr = df_pbr.copy()

# 모든 기업에 대해 1/PBR 계산
for company in company_columns:
    df_inverse_pbr[company] = 1 / df_inverse_pbr[company]

# 결과 출력
df_inverse_pbr

In [ ]:
num_company = len(company_columns)

top_pbr_year = pd.DataFrame()

for year in range(2001, 2024):
    top_companies = kse_functions.top_companies_by_inverse_Rank(df_inverse_pbr, year, num_company) # 매개변수를 조절함으로 조절 가능
    top_companies['Year'] = year
    top_companies['Rank'] = range(1, len(top_companies) + 1)
    top_pbr_year = pd.concat([top_pbr_year, top_companies], ignore_index=True)

# 결과 출력
top_pbr_year

In [ ]:
close_1 = df_close.copy()

# 첫 번째 열을 인덱스로 설정
close_1.set_index(close_1.columns[0], inplace=True)

# 인덱스를 날짜 형식으로 변환
close_1.index = pd.to_datetime(close_1.index)

# 초기 설정
sale_month_imsi = 3
sale_year_imsi = 2002
purchase_month_imsi = 4 
purchase_year_imsi = 2001

# 시작 연도와 종료 연도 설정
start_year = 2001
end_year = 2022

# 각 연도에 대한 처리
total_values = {}
profits = {}

for year in range(start_year, end_year + 1):

    top_pbr_current_year = top_pbr_year[top_pbr_year['Year'] == year]
    Purchase_current = kse_functions.Price_for_Purchase_Sale(purchase_year_imsi + (year - start_year), purchase_month_imsi, 
                                               sale_year_imsi + (year - start_year), sale_month_imsi,
                                               top_pbr_current_year, close_1)

    if year == start_year:
        share_current = kse_functions.Purchase_share(Purchase_current, top_pbr_current_year, 5000000 * len(top_pbr_current_year))
    else:
        share_current = kse_functions.Purchase_share(Purchase_current, top_pbr_current_year, 5000000 * len(top_pbr_current_year))
        ## 추후 백테스팅할때는 이부분을 수정 5000000*num_company -> total_values[year - 1]
    Purchase_current_first = Purchase_current.drop_duplicates(subset=['기업명', '매수일'], keep='first')
    Purchase_current_share = pd.merge(share_current, Purchase_current_first, on='기업명', how='left')

    # 잔금 계산 및 '잔금' 컬럼 추가
    # 잔금은 총 매출에 포함되어 있으므로 전체 투자원금을 차감합니다.
    Purchase_current_share = add_price_direction_targets(recalculate_returns(Purchase_current_share))

    # 포트폴리오의 총 가치 계산
    total_value_current = Purchase_current_share['매출'].sum()
    total_values[year] = total_value_current
    profits[year] = Purchase_current_share

# 결과 출력 (필요에 따라 조정)
for year, profit in profits.items():
    print(f"Year: {year}, Total Value: {total_values[year]}")
    print(profit)

In [ ]:
# 각 연도에 대한 처리
for year in range(start_year, end_year + 1):
     # 연도별 Top 20 주식 선정
    top_pbr_current_year = top_pbr_year[top_pbr_year['Year'] == year]
    
    # 구매 및 판매 가격 계산
    Purchase_current = kse_functions.Price_for_Purchase_Sale(
        purchase_year_imsi + (year - start_year), purchase_month_imsi, 
        sale_year_imsi + (year - start_year), sale_month_imsi,
        top_pbr_current_year, close_1
    )

    # 주식 분배 계산
    if year == start_year:
        share_current = kse_functions.Purchase_share(Purchase_current, top_pbr_current_year, 5000000 * len(top_pbr_current_year))
    else:
        share_current = kse_functions.Purchase_share(Purchase_current, top_pbr_current_year, 5000000 * len(top_pbr_current_year))

    # 구매 데이터 정리
    Purchase_current_first = Purchase_current.drop_duplicates(subset=['기업명', '매수일'], keep='first')
    Purchase_current_share = pd.merge(share_current, Purchase_current_first, on='기업명', how='left')

    # 잔금 및 수익 계산
    # 잔금은 총 매출에 포함되어 있으므로 전체 투자원금을 차감합니다.
    Purchase_current_share = add_price_direction_targets(recalculate_returns(Purchase_current_share))

    # 포트폴리오의 총 가치 계산 및 저장
    total_value_current = Purchase_current_share['매출'].sum()
    total_values[year] = total_value_current
    profits[year] = Purchase_current_share
    # 연도별 데이터 프레임 저장 확인
    if isinstance(Purchase_current_share, pd.DataFrame):
        profits[year] = Purchase_current_share
    else:
        print(f"Year {year} did not produce a valid DataFrame")

# 각 연도별 데이터 프레임에 '연도' 열 추가 및 리스트에 저장
all_profits = []
for year, profit_df in profits.items():
    if isinstance(profit_df, pd.DataFrame):
        profit_df['연도'] = year
        all_profits.append(profit_df)
    else:
        print(f"Invalid DataFrame for year {year}")

# 모든 연도의 데이터 프레임을 하나로 결합
if all_profits:
    combined_profit = pd.concat(all_profits)
    # 결과 출력
    print(combined_profit)
else:
    print("No valid DataFrames to combine")

In [ ]:
output_filename = "PBR_profit_for_tree.xlsx"
combined_profit.to_excel(output_filename, index=False)

print(f"Data saved to {output_filename}")

In [ ]:
df_pbr_profit = add_price_direction_targets(recalculate_returns(pd.read_excel("PBR_profit_for_tree.xlsx")))

def merge_pbr_values(df_profit, df_inverse_pbr):
    # 새로운 데이터프레임 생성
    df_merged = df_profit.copy()

    # 연도와 기업명을 기준으로 1/PBR 값을 매핑
    for index, row in df_profit.iterrows():
        year = row['연도']
        company = row['기업명']
        
        # df_inverse_per에서 해당 연도와 기업의 PBR 값을 찾음
        pbr_value = df_inverse_pbr.loc[df_inverse_pbr['Year'] == year, company].iloc[0]

        # 새로운 열에 값을 추가
        df_merged.at[index, '1/PBR'] = pbr_value

    return df_merged

df_merged = merge_pbr_values(df_pbr_profit, df_inverse_pbr)

# 원본 데이터프레임의 복사본을 생성
df_pbr_profit_cleaned = df_merged.copy()

# NaN 값이 있는 행을 제거
df_pbr_profit_cleaned.dropna(subset=['1/PBR'], inplace=True)

# 1/PBR 값의 순위를 내림차순으로 계산
df_pbr_profit_cleaned['1/PBR Rank'] = df_pbr_profit_cleaned.groupby('연도')['1/PBR'].rank(ascending=False)

output_filename = "PBR_profit_final.xlsx"
df_pbr_profit_cleaned.to_excel(output_filename, index=False)

print(f"Data saved to {output_filename}")

In [ ]:
### PSR

In [ ]:
# Name과 SPS 컬럼을 추출하고, 회계년으로 정렬
SPS_Pure_df = df_fin[['Name', '회계년', 'SPS']].sort_values(by=['회계년'])

# 회계년을 Year로 통일하여 전처리
SPS_Pure_df.rename(columns={'회계년': 'Year'}, inplace=True)

# SPS 데이터를 연도별로 재구조화
SPS_Pure_df_pivot = SPS_Pure_df.pivot(index='Year', columns='Name', values='SPS')

# PSR 데이터 프레임 초기화
df_psr = pd.DataFrame()

# 연도별로 PSR 계산
for year in range(SPS_Pure_df['Year'].min() + 1, average_prices_df['Year'].max() + 1):
    if year in average_prices_df['Year'].values and (year - 1) in SPS_Pure_df_pivot.index:
        previous_year_sps = SPS_Pure_df_pivot.loc[year - 1]  # 이전 연도의 SPS
        current_year_prices = average_prices_df[average_prices_df['Year'] == year].iloc[0]  # 해당 연도의 평균 주가

        # PSR 계산
        psr_data = {'Year': year}
        for company in company_columns:
            sps = previous_year_sps.get(company)
            price = current_year_prices.get(company)
            if sps and price and sps != 0:
                psr = price / sps
            else:
                psr = None  # SPS가 0이거나 없는 경우 PSR 계산 불가
            psr_data[company] = psr

        # PSR 데이터 추가
        df_psr = pd.concat([df_psr, pd.DataFrame([psr_data])], ignore_index=True)

# Year의 형을 정수형로 바꾸기
df_psr['Year'] = df_psr['Year'].astype(int)

# df_psr 데이터 프레임의 복사본 생성
df_inverse_psr = df_psr.copy()

# 모든 기업에 대해 1/PSR 계산
for company in company_columns:
    df_inverse_psr[company] = 1 / df_inverse_psr[company]
    
num_company = len(company_columns)

top_psr_year = pd.DataFrame()

for year in range(2001, 2024):
    top_companies = kse_functions.top_companies_by_inverse_Rank(df_inverse_psr, year, num_company) # 매개변수를 조절함으로 조절 가능
    top_companies['Year'] = year
    top_companies['Rank'] = range(1, len(top_companies) + 1)
    top_psr_year = pd.concat([top_psr_year, top_companies], ignore_index=True)
    
# 각 연도에 대한 처리
total_values = {}
profits = {}

for year in range(start_year, end_year + 1):

    top_psr_current_year = top_psr_year[top_psr_year['Year'] == year]
    Purchase_current = kse_functions.Price_for_Purchase_Sale(purchase_year_imsi + (year - start_year), purchase_month_imsi, 
                                               sale_year_imsi + (year - start_year), sale_month_imsi,
                                               top_psr_current_year, close_1)

    if year == start_year:
        share_current = kse_functions.Purchase_share(Purchase_current, top_psr_current_year, 5000000 * len(top_psr_current_year))
    else:
        share_current = kse_functions.Purchase_share(Purchase_current, top_psr_current_year, 5000000 * len(top_psr_current_year))
        ## 추후 백테스팅할때는 이부분을 수정 5000000*num_company -> total_values[year - 1]
    Purchase_current_first = Purchase_current.drop_duplicates(subset=['기업명', '매수일'], keep='first')
    Purchase_current_share = pd.merge(share_current, Purchase_current_first, on='기업명', how='left')

    # 잔금 계산 및 '잔금' 컬럼 추가
    # 잔금은 총 매출에 포함되어 있으므로 전체 투자원금을 차감합니다.
    Purchase_current_share = add_price_direction_targets(recalculate_returns(Purchase_current_share))

    # 포트폴리오의 총 가치 계산
    total_value_current = Purchase_current_share['매출'].sum()
    total_values[year] = total_value_current
    profits[year] = Purchase_current_share

# 각 연도에 대한 처리
for year in range(start_year, end_year + 1):
    
    top_psr_current_year = top_psr_year[top_psr_year['Year'] == year]
    
    # 구매 및 판매 가격 계산
    Purchase_current = kse_functions.Price_for_Purchase_Sale(
        purchase_year_imsi + (year - start_year), purchase_month_imsi, 
        sale_year_imsi + (year - start_year), sale_month_imsi,
        top_psr_current_year, close_1
    )

    # 주식 분배 계산
    if year == start_year:
        share_current = kse_functions.Purchase_share(Purchase_current, top_psr_current_year, 5000000 * len(top_psr_current_year))
    else:
        share_current = kse_functions.Purchase_share(Purchase_current, top_psr_current_year, 5000000 * len(top_psr_current_year))

    # 구매 데이터 정리
    Purchase_current_first = Purchase_current.drop_duplicates(subset=['기업명', '매수일'], keep='first')
    Purchase_current_share = pd.merge(share_current, Purchase_current_first, on='기업명', how='left')

    # 잔금 및 수익 계산
    # 잔금은 총 매출에 포함되어 있으므로 전체 투자원금을 차감합니다.
    Purchase_current_share = add_price_direction_targets(recalculate_returns(Purchase_current_share))

    # 포트폴리오의 총 가치 계산 및 저장
    total_value_current = Purchase_current_share['매출'].sum()
    total_values[year] = total_value_current
    profits[year] = Purchase_current_share
    # 연도별 데이터 프레임 저장 확인
    if isinstance(Purchase_current_share, pd.DataFrame):
        profits[year] = Purchase_current_share
    else:
        print(f"Year {year} did not produce a valid DataFrame")

# 각 연도별 데이터 프레임에 '연도' 열 추가 및 리스트에 저장
all_profits = []
for year, profit_df in profits.items():
    if isinstance(profit_df, pd.DataFrame):
        profit_df['연도'] = year
        all_profits.append(profit_df)
    else:
        print(f"Invalid DataFrame for year {year}")

# 모든 연도의 데이터 프레임을 하나로 결합
if all_profits:
    combined_profit = pd.concat(all_profits)
    # 결과 출력
    print(combined_profit)
else:
    print("No valid DataFrames to combine")
    
output_filename = "PSR_profit_for_tree.xlsx"
combined_profit.to_excel(output_filename, index=False)

print(f"Data saved to {output_filename}")

In [ ]:
df_psr_profit = add_price_direction_targets(recalculate_returns(pd.read_excel("PSR_profit_for_tree.xlsx")))

def merge_psr_values(df_profit, df_inverse_psr):
    # 새로운 데이터프레임 생성
    df_merged = df_profit.copy()

    # 연도와 기업명을 기준으로 1/PBR 값을 매핑
    for index, row in df_profit.iterrows():
        year = row['연도']
        company = row['기업명']
        
        # df_inverse_psr에서 해당 연도와 기업의 PSR 값을 찾음
        psr_value = df_inverse_psr.loc[df_inverse_psr['Year'] == year, company].iloc[0]

        # 새로운 열에 값을 추가
        df_merged.at[index, '1/PSR'] = psr_value

    return df_merged

df_merged = merge_psr_values(df_psr_profit, df_inverse_psr)

# 원본 데이터프레임의 복사본을 생성
df_psr_profit_cleaned = df_merged.copy()

# NaN 값이 있는 행을 제거
df_psr_profit_cleaned.dropna(subset=['1/PSR'], inplace=True)

# 1/PSR 값의 순위를 내림차순으로 계산
df_psr_profit_cleaned['1/PSR Rank'] = df_psr_profit_cleaned.groupby('연도')['1/PSR'].rank(ascending=False)

output_filename = "PSR_profit_final.xlsx"
df_psr_profit_cleaned.to_excel(output_filename, index=False)

print(f"Data saved to {output_filename}")

In [ ]:
### PCR

In [ ]:
# Name과 CFPS 컬럼을 추출하고, 회계년으로 정렬
CFPS_Pure_df = df_fin[['Name', '회계년', 'CFPS']].sort_values(by=['회계년'])

# 회계년을 Year로 통일하여 전처리
CFPS_Pure_df.rename(columns={'회계년': 'Year'}, inplace=True)

# CFPS 데이터를 연도별로 재구조화
CFPS_Pure_df_pivot = CFPS_Pure_df.pivot(index='Year', columns='Name', values='CFPS')

# PCR 데이터 프레임 초기화
df_pcr = pd.DataFrame()

# 연도별로 PCR 계산
for year in range(CFPS_Pure_df['Year'].min() + 1, average_prices_df['Year'].max() + 1):
    if year in average_prices_df['Year'].values and (year - 1) in CFPS_Pure_df_pivot.index:
        previous_year_cfps = CFPS_Pure_df_pivot.loc[year - 1]  # 이전 연도의 CFPS
        current_year_prices = average_prices_df[average_prices_df['Year'] == year].iloc[0]  # 해당 연도의 평균 주가

        # PCR 계산
        pcr_data = {'Year': year}
        for company in company_columns:
            cfps = previous_year_cfps.get(company)
            price = current_year_prices.get(company)
            if cfps and price and cfps != 0:
                pcr = price / cfps
            else:
                pcr = None  # CFPS가 0이거나 없는 경우 PCR 계산 불가
            pcr_data[company] = pcr

        # PCR 데이터 추가
        df_pcr = pd.concat([df_pcr, pd.DataFrame([pcr_data])], ignore_index=True)

# Year의 형을 정수형로 바꾸기
df_pcr['Year'] = df_pcr['Year'].astype(int)

# df_pcr 데이터 프레임의 복사본 생성
df_inverse_pcr = df_pcr.copy()

# 모든 기업에 대해 1/PCR 계산
for company in company_columns:
    df_inverse_pcr[company] = 1 / df_inverse_pcr[company]
    
num_company = len(company_columns)

top_pcr_year = pd.DataFrame()

for year in range(2001, 2024):
    top_companies = kse_functions.top_companies_by_inverse_Rank(df_inverse_pcr, year, num_company) # 매개변수를 조절함으로 조절 가능
    top_companies['Year'] = year
    top_companies['Rank'] = range(1, len(top_companies) + 1)
    top_pcr_year = pd.concat([top_pcr_year, top_companies], ignore_index=True)
    
# 각 연도에 대한 처리
total_values = {}
profits = {}

for year in range(start_year, end_year + 1):

    top_pcr_current_year = top_pcr_year[top_pcr_year['Year'] == year]
    Purchase_current = kse_functions.Price_for_Purchase_Sale(purchase_year_imsi + (year - start_year), purchase_month_imsi, 
                                               sale_year_imsi + (year - start_year), sale_month_imsi,
                                               top_pcr_current_year, close_1)

    if year == start_year:
        share_current = kse_functions.Purchase_share(Purchase_current, top_pcr_current_year, 5000000 * len(top_pcr_current_year))
    else:
        share_current = kse_functions.Purchase_share(Purchase_current, top_pcr_current_year, 5000000 * len(top_pcr_current_year))
        ## 추후 백테스팅할때는 이부분을 수정 5000000*num_company -> total_values[year - 1]
    Purchase_current_first = Purchase_current.drop_duplicates(subset=['기업명', '매수일'], keep='first')
    Purchase_current_share = pd.merge(share_current, Purchase_current_first, on='기업명', how='left')

    # 잔금 계산 및 '잔금' 컬럼 추가
    # 잔금은 총 매출에 포함되어 있으므로 전체 투자원금을 차감합니다.
    Purchase_current_share = add_price_direction_targets(recalculate_returns(Purchase_current_share))

    # 포트폴리오의 총 가치 계산
    total_value_current = Purchase_current_share['매출'].sum()
    total_values[year] = total_value_current
    profits[year] = Purchase_current_share

# 각 연도에 대한 처리
for year in range(start_year, end_year + 1):
     
    top_pcr_current_year = top_pcr_year[top_pcr_year['Year'] == year]
    
    # 구매 및 판매 가격 계산
    Purchase_current = kse_functions.Price_for_Purchase_Sale(
        purchase_year_imsi + (year - start_year), purchase_month_imsi, 
        sale_year_imsi + (year - start_year), sale_month_imsi,
        top_pcr_current_year, close_1
    )

    # 주식 분배 계산
    if year == start_year:
        share_current = kse_functions.Purchase_share(Purchase_current, top_pcr_current_year, 5000000 * len(top_pcr_current_year))
    else:
        share_current = kse_functions.Purchase_share(Purchase_current, top_pcr_current_year, 5000000 * len(top_pcr_current_year))

    # 구매 데이터 정리
    Purchase_current_first = Purchase_current.drop_duplicates(subset=['기업명', '매수일'], keep='first')
    Purchase_current_share = pd.merge(share_current, Purchase_current_first, on='기업명', how='left')

    # 잔금 및 수익 계산
    # 잔금은 총 매출에 포함되어 있으므로 전체 투자원금을 차감합니다.
    Purchase_current_share = add_price_direction_targets(recalculate_returns(Purchase_current_share))

    # 포트폴리오의 총 가치 계산 및 저장
    total_value_current = Purchase_current_share['매출'].sum()
    total_values[year] = total_value_current
    profits[year] = Purchase_current_share
    # 연도별 데이터 프레임 저장 확인
    if isinstance(Purchase_current_share, pd.DataFrame):
        profits[year] = Purchase_current_share
    else:
        print(f"Year {year} did not produce a valid DataFrame")

# 각 연도별 데이터 프레임에 '연도' 열 추가 및 리스트에 저장
all_profits = []
for year, profit_df in profits.items():
    if isinstance(profit_df, pd.DataFrame):
        profit_df['연도'] = year
        all_profits.append(profit_df)
    else:
        print(f"Invalid DataFrame for year {year}")

# 모든 연도의 데이터 프레임을 하나로 결합
if all_profits:
    combined_profit = pd.concat(all_profits)
    # 결과 출력
    print(combined_profit)
else:
    print("No valid DataFrames to combine")
    
output_filename = "PCR_profit_for_tree.xlsx"
combined_profit.to_excel(output_filename, index=False)

print(f"Data saved to {output_filename}")

In [ ]:
df_pcr_profit = add_price_direction_targets(recalculate_returns(pd.read_excel("PCR_profit_for_tree.xlsx")))

def merge_pcr_values(df_profit, df_inverse_pcr):
    # 새로운 데이터프레임 생성
    df_merged = df_profit.copy()

    # 연도와 기업명을 기준으로 1/PCR 값을 매핑
    for index, row in df_profit.iterrows():
        year = row['연도']
        company = row['기업명']
        
        # df_inverse_pcr에서 해당 연도와 기업의 PCR 값을 찾음
        pcr_value = df_inverse_pcr.loc[df_inverse_pcr['Year'] == year, company].iloc[0]

        # 새로운 열에 값을 추가
        df_merged.at[index, '1/PCR'] = pcr_value

    return df_merged

df_merged = merge_pcr_values(df_pcr_profit, df_inverse_pcr)

# 원본 데이터프레임의 복사본을 생성
df_pcr_profit_cleaned = df_merged.copy()

# NaN 값이 있는 행을 제거
df_pcr_profit_cleaned.dropna(subset=['1/PCR'], inplace=True)

# 1/PCR 값의 순위를 내림차순으로 계산
df_pcr_profit_cleaned['1/PCR Rank'] = df_pcr_profit_cleaned.groupby('연도')['1/PCR'].rank(ascending=False)

output_filename = "PCR_profit_final.xlsx"
df_pcr_profit_cleaned.to_excel(output_filename, index=False)

print(f"Data saved to {output_filename}")

In [ ]:
### 동일비중 전략

In [ ]:
# 모든 지표 데이터 프레임의 리스트
data_frames = [df_inverse_per, df_inverse_pbr, df_inverse_psr, df_inverse_pcr]

df_inverse_all = kse_functions.calculate_weighted_average(data_frames)

num_company = len(company_columns)

top_all_year = pd.DataFrame()

for year in range(2001, 2024):
    top_companies = kse_functions.top_companies_by_inverse_Rank(df_inverse_all, year, num_company) # 매개변수를 조절함으로 조절 가능
    top_companies['Year'] = year
    top_companies['Rank'] = range(1, len(top_companies) + 1)
    top_all_year = pd.concat([top_all_year, top_companies], ignore_index=True)
    
# 각 연도에 대한 처리
total_values = {}
profits = {}

for year in range(start_year, end_year + 1):

    top_all_current_year = top_all_year[top_all_year['Year'] == year]
    Purchase_current = kse_functions.Price_for_Purchase_Sale(purchase_year_imsi + (year - start_year), purchase_month_imsi, 
                                               sale_year_imsi + (year - start_year), sale_month_imsi,
                                               top_all_current_year, close_1)

    if year == start_year:
        share_current = kse_functions.Purchase_share(Purchase_current, top_all_current_year, 5000000 * len(top_all_current_year))
    else:
        share_current = kse_functions.Purchase_share(Purchase_current, top_all_current_year, 5000000 * len(top_all_current_year))
        ## 추후 백테스팅할때는 이부분을 수정 5000000*num_company -> total_values[year - 1]
    Purchase_current_first = Purchase_current.drop_duplicates(subset=['기업명', '매수일'], keep='first')
    Purchase_current_share = pd.merge(share_current, Purchase_current_first, on='기업명', how='left')

    # 잔금 계산 및 '잔금' 컬럼 추가
    # 잔금은 총 매출에 포함되어 있으므로 전체 투자원금을 차감합니다.
    Purchase_current_share = add_price_direction_targets(recalculate_returns(Purchase_current_share))

    # 포트폴리오의 총 가치 계산
    total_value_current = Purchase_current_share['매출'].sum()
    total_values[year] = total_value_current
    profits[year] = Purchase_current_share

# 각 연도에 대한 처리
for year in range(start_year, end_year + 1):
     
    top_all_current_year = top_all_year[top_all_year['Year'] == year]
    
    # 구매 및 판매 가격 계산
    Purchase_current = kse_functions.Price_for_Purchase_Sale(
        purchase_year_imsi + (year - start_year), purchase_month_imsi, 
        sale_year_imsi + (year - start_year), sale_month_imsi,
        top_all_current_year, close_1
    )

    # 주식 분배 계산
    if year == start_year:
        share_current = kse_functions.Purchase_share(Purchase_current, top_all_current_year, 5000000 * len(top_all_current_year))
    else:
        share_current = kse_functions.Purchase_share(Purchase_current, top_all_current_year, 5000000 * len(top_all_current_year))

    # 구매 데이터 정리
    Purchase_current_first = Purchase_current.drop_duplicates(subset=['기업명', '매수일'], keep='first')
    Purchase_current_share = pd.merge(share_current, Purchase_current_first, on='기업명', how='left')

    # 잔금 및 수익 계산
    # 잔금은 총 매출에 포함되어 있으므로 전체 투자원금을 차감합니다.
    Purchase_current_share = add_price_direction_targets(recalculate_returns(Purchase_current_share))

    # 포트폴리오의 총 가치 계산 및 저장
    total_value_current = Purchase_current_share['매출'].sum()
    total_values[year] = total_value_current
    profits[year] = Purchase_current_share
    # 연도별 데이터 프레임 저장 확인
    if isinstance(Purchase_current_share, pd.DataFrame):
        profits[year] = Purchase_current_share
    else:
        print(f"Year {year} did not produce a valid DataFrame")

# 각 연도별 데이터 프레임에 '연도' 열 추가 및 리스트에 저장
all_profits = []
for year, profit_df in profits.items():
    if isinstance(profit_df, pd.DataFrame):
        profit_df['연도'] = year
        all_profits.append(profit_df)
    else:
        print(f"Invalid DataFrame for year {year}")

# 모든 연도의 데이터 프레임을 하나로 결합
if all_profits:
    combined_profit = pd.concat(all_profits)
    # 결과 출력
    print(combined_profit)
else:
    print("No valid DataFrames to combine")
    
output_filename = "ALL_profit_for_tree.xlsx"
combined_profit.to_excel(output_filename, index=False)

print(f"Data saved to {output_filename}")

In [ ]:
df_all_profit = add_price_direction_targets(recalculate_returns(pd.read_excel("ALL_profit_for_tree.xlsx")))

df_merged = merge_per_values(df_all_profit, df_inverse_per)
df_merged = merge_pbr_values(df_merged, df_inverse_pbr)
df_merged = merge_psr_values(df_merged, df_inverse_psr)
df_merged = merge_pcr_values(df_merged, df_inverse_pcr)

def merge_all_values(df_profit, df_inverse_all):
    # 새로운 데이터프레임 생성
    df_merged = df_profit.copy()

    # 연도와 기업명을 기준으로 값을 매핑
    for index, row in df_profit.iterrows():
        year = row['연도']
        company = row['기업명']
        
        # df_inverse_all에서 해당 연도와 기업의 값을 찾음
        all_value = df_inverse_all.loc[df_inverse_all['Year'] == year, company].iloc[0]

        # 새로운 열에 값을 추가
        df_merged.at[index, 'ALL'] = all_value

    return df_merged

df_merged = merge_all_values(df_merged, df_inverse_all)

# 원본 데이터프레임의 복사본을 생성
df_all_profit_cleaned = df_merged.copy()

# NaN 값이 있는 행을 제거
df_all_profit_cleaned.dropna(subset=['ALL'], inplace=True)

# 값의 순위를 내림차순으로 계산
df_all_profit_cleaned['ALL Rank'] = df_all_profit_cleaned.groupby('연도')['ALL'].rank(ascending=False)

output_filename = "ALL_profit_final.xlsx"
df_all_profit_cleaned.to_excel(output_filename, index=False)

print(f"Data saved to {output_filename}")

In [ ]:
### 네가지 지표를 모두 활용하는 경우 지표별 순위를 기준으로 측정

In [ ]:
df_all_profit_cleaned['1/PER Rank'] = df_all_profit_cleaned.groupby('연도')['1/PER'].rank(ascending=False)
df_all_profit_cleaned['1/PBR Rank'] = df_all_profit_cleaned.groupby('연도')['1/PBR'].rank(ascending=False)
df_all_profit_cleaned['1/PSR Rank'] = df_all_profit_cleaned.groupby('연도')['1/PSR'].rank(ascending=False)
df_all_profit_cleaned['1/PCR Rank'] = df_all_profit_cleaned.groupby('연도')['1/PCR'].rank(ascending=False)
# 값의 순위를 내림차순으로 계산
df_all_profit_cleaned['ALL Rank'] = df_all_profit_cleaned.groupby('연도')['ALL'].rank(ascending=False)

df_all_profit_cleaned

In [ ]:
output_filename = "ALL_profit_final(개).xlsx"
df_all_profit_cleaned.to_excel(output_filename, index=False)

print(f"Data saved to {output_filename}")